In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout, Bidirectional, Conv1D, MaxPooling1D, BatchNormalization
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ==============================================================================
# 1. LECTURA DE DATOS AUMENTADOS Y EXTRACCIÓN DE SECUENCIAS (6s)
# ==============================================================================
PROCESSED_AUGMENTED_PATH = Path("../Data/Processed Data/Augmented")
archivos_aumentados = list(PROCESSED_AUGMENTED_PATH.glob("*.csv"))
print(f"Archivos localizados: {len(archivos_aumentados)}")

WINDOW_SIZE = 121 # 6 segundos a 20Hz
FEATURES_LSTM = ["baro_altitude", "accl_x", "accl_y", "accl_z", "At", "P"]

X_sequences, y_sequences, base_flight_ids = [], [], []

for archivo in archivos_aumentados:
    df = pd.read_csv(archivo)
    
    # Extraer el ID base para evitar Fuga de Datos (Data Leakage)
    # Separa "10_standardized_sintetico_001.csv" a -> "10_standardized"
    nombre = archivo.name
    base_id = nombre.split("_base")[0].split("_sintetico")[0]
    
    df = df.sort_values("time")
    window = df.iloc[:WINDOW_SIZE]
    
    # Omitir si no cumple con los 6 segundos de telemetría requerida
    if len(window) != WINDOW_SIZE:
        continue
        
    # El apogeo real es la altitud máxima alcanzada en todo el archivo
    apogee = df["baro_altitude"].max()
    
    X_sequences.append(window[FEATURES_LSTM].values)
    y_sequences.append(apogee)
    base_flight_ids.append(base_id)

X_seq = np.array(X_sequences, dtype=np.float32)
y_seq = np.array(y_sequences, dtype=np.float32)
base_flight_ids = np.array(base_flight_ids)
print(f"Secuencias extraídas: X={X_seq.shape}, y={y_seq.shape}")

# ==============================================================================
# 2. PARTICIÓN DE RED SEGURA (TRAIN/VAL/TEST AGRUPADO POR VUELO BASE)
# ==============================================================================
unique_base_ids = np.unique(base_flight_ids)
# 60% Train, 20% Val, 20% Test (pero a nivel de ID ORIGINAL)
train_ids, temp_ids = train_test_split(unique_base_ids, test_size=0.40, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=42)

train_mask = np.isin(base_flight_ids, train_ids)
val_mask = np.isin(base_flight_ids, val_ids)
test_mask = np.isin(base_flight_ids, test_ids)

X_train, y_train = X_seq[train_mask], y_seq[train_mask]
X_val, y_val = X_seq[val_mask], y_seq[val_mask]
X_test, y_test = X_seq[test_mask], y_seq[test_mask]

print(f"Muestras Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

# ==============================================================================
# 3. ESCALAMIENTO ROBUSTO 3D (FEATURES) y 1D (TARGET)
# ==============================================================================
n_features = X_train.shape[2]
scaler_X = StandardScaler()

# Aplanar a 2D para ajustar, luego regresar a 3D
X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
X_val_scaled = scaler_X.transform(X_val.reshape(-1, n_features)).reshape(X_val.shape)
X_test_scaled = scaler_X.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

# ==============================================================================
# 4. CONSTRUCCIÓN DEL MODELO HÍBRIDO (LSTM V5: Conv1D + BiLSTM)
# ==============================================================================
def build_lstm_v5(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        # Extrae picos locales y filtra ruido residual del Data Augmentation
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        
        # Lee la inercia del cohete de inicio a fin y de fin a inicio
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(64, return_sequences=False)),
        Dropout(0.3),
        
        # Regularización L2 fuerte para evitar Overfitting
        Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.005)),
        Dense(16, activation='relu'),
        Dense(1) 
    ], name="LSTM_V5_Hibrido")

    # Clipnorm = 1.0 evita la explosión de gradientes (mantiene el MSE bajo control)
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0)
    
    # Pérdida de Huber para inmunidad a outliers sintéticos
    model.compile(
        optimizer=optimizer,
        loss=Huber(delta=1.0),
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

modelo_v5 = build_lstm_v5(input_shape=(WINDOW_SIZE, n_features))
modelo_v5.summary()

# ==============================================================================
# 5. ENTRENAMIENTO AUTOMATIZADO CON CALLBACKS
# ==============================================================================
early_stop = EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-6)

print("\nEntrenando LSTM V5...")
history_v5 = modelo_v5.fit(
    X_train_scaled, y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=200, 
    batch_size=32, 
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ==============================================================================
# 6. EVALUACIÓN Y GRÁFICA (Resultados devueltos a escala en METROS)
# ==============================================================================
# Predecir sobre Validación y revertir el Standard Scaler
pred_val_scaled = modelo_v5.predict(X_val_scaled, verbose=0).ravel()
y_pred_metros = scaler_y.inverse_transform(pred_val_scaled.reshape(-1, 1)).ravel()
y_true_metros = y_val  # El apogeo real crudo en metros

mae = mean_absolute_error(y_true_metros, y_pred_metros)
mse = mean_squared_error(y_true_metros, y_pred_metros)
r2 = r2_score(y_true_metros, y_pred_metros)

print("="*50)
print("MÉTRICAS FINALES DE VALIDACIÓN (LSTM V5) EN METROS")
print("="*50)
print(f"MAE (Error Absoluto Medio) : {mae:.2f} metros")
print(f"MSE (Error Cuadrático Medio): {mse:.2f}")
print(f"RMSE (Raíz del MSE)        : {np.sqrt(mse):.2f} metros")
print(f"R2 Score                   : {r2:.4f}")
print("="*50)

# Gráfica de Pérdida
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_v5.history["loss"], label="Train Loss (Huber)")
axes[0].plot(history_v5.history["val_loss"], label="Val Loss (Huber)")
axes[0].set_title("Curva de Aprendizaje - LSTM V5")
axes[0].set_xlabel("Épocas")
axes[0].set_ylabel("Pérdida (Loss)")
axes[0].legend()
axes[0].grid(True)

# Gráfica Real vs Predicción
lo = min(np.min(y_true_metros), np.min(y_pred_metros))
hi = max(np.max(y_true_metros), np.max(y_pred_metros))
axes[1].scatter(y_true_metros, y_pred_metros, alpha=0.6, color='blue', edgecolor='k')
axes[1].plot([lo, hi], [lo, hi], 'k--', label="Predicción Perfecta (y=x)")
axes[1].set_title("Apogeo: Real vs. Predicho (Validación)")
axes[1].set_xlabel("Apogeo Real [m]")
axes[1].set_ylabel("Apogeo Predicho LSTM [m]")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()